In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import shutil
import glob
from PIL import Image

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.append(ROOT)
os.chdir(ROOT)

from config import Config
from model import SingleStreamDiT
from latents import decode_latents_to_image, prepare_latents_for_decode
from samplers import run_sampling_pipeline
from model_loader import load_vae

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
Config.device = DEVICE
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Jupyter environment configured on device: {DEVICE}")

In [ ]:
FILENAME = "ema_epoch_1500.pt"   
latest_checkpoint = os.path.join(Config.checkpoint_dir, FILENAME)
CH_BASE_NAME = os.path.splitext(FILENAME)[0]

print(f"Loading EMA checkpoint: {os.path.basename(latest_checkpoint)}")

print("Loading SingleStreamDiT backbone...")
model = SingleStreamDiT(
    in_channels=Config.in_channels,
    patch_size=Config.patch_size,
    hidden_size=Config.hidden_size,
    depth=Config.depth,
    num_heads=Config.num_heads,
    text_embed_dim=Config.text_embed_dim,
    refiner_depth=Config.refiner_depth,
).to(DEVICE, Config.dtype)

checkpoint_data = torch.load(latest_checkpoint, map_location=DEVICE)

model.load_state_dict(checkpoint_data)
model.eval()

print("Loading VAE model...")
vae = load_vae()
print("System successfully initialized and loaded EMA weights!")

In [ ]:
target_filename = "10003724.pt"
target_file = os.path.join(Config.cache_dir, target_filename)  

if not os.path.exists(target_file):
    raise FileNotFoundError(f"Target cache file not found at: {target_file}")

print(f"Extracting target conditions from: {target_filename}")

data = torch.load(target_file, map_location=DEVICE)
h, w = data["height"], data["width"]

if "text_embeds_list" in data:
    text_embeds = data["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype)
    text_mask = data["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
else:
    text_embeds = data["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype)
    text_mask = data["attention_mask"].unsqueeze(0).to(DEVICE)

uncond_embeds = torch.zeros_like(text_embeds)
uncond_mask = torch.ones_like(text_mask)
combined_text_embeds = torch.cat([uncond_embeds, text_embeds], dim=0)
combined_mask = torch.cat([uncond_mask, text_mask], dim=0)

torch_generator = torch.Generator(device=DEVICE).manual_seed(42)
initial_noise = torch.randn(1, Config.in_channels, h // Config.vae_downsample_factor, 
                            w // Config.vae_downsample_factor, generator=torch_generator, 
                            device=DEVICE, dtype=Config.dtype)

print(f"Successfully loaded conditions. Image dimensions: {w}x{h}")

In [ ]:
print("Generating CONTROL image with Conv2D layers ACTIVE...")
with torch.no_grad():
    with torch.amp.autocast("cuda", dtype=Config.dtype):
        latents_control = run_sampling_pipeline(model=model, initial_noise=initial_noise.clone(), 
                                                steps=30, combined_text_embeds=combined_text_embeds, 
                                                cfg=1.0, text_mask=combined_mask, sampler_type="euler",
                                                scheduler_type="uniform", shift_val=2.5)

img_control = decode_latents_to_image(vae, latents_control, DEVICE)
print("Control image successfully generated and held in memory.")

In [ ]:
original_convs = {}
for name, block_list in [("blocks", model.blocks), ("noise_refiner", model.noise_refiner)]:
    for i, block in enumerate(block_list):
        if hasattr(block, 'ffn') and hasattr(block.ffn, 'dwconv'):
            original_convs[(name, i)] = block.ffn.dwconv
            block.ffn.dwconv = nn.Identity()

print("Generating EXPERIMENTAL image with Conv2D layers INACTIVE...")
with torch.no_grad():
    with torch.amp.autocast("cuda", dtype=Config.dtype):
        latents_ablated = run_sampling_pipeline(model=model, initial_noise=initial_noise.clone(), 
                                                steps=30, combined_text_embeds=combined_text_embeds, 
                                                cfg=1.0, text_mask=combined_mask, sampler_type="euler", 
                                                scheduler_type="uniform", shift_val=2.5)

img_ablated = decode_latents_to_image(vae, latents_ablated, DEVICE)

print("Restoring original Conv2D layers...")
for (name, i), conv in original_convs.items():
    getattr(model, name)[i].ffn.dwconv = conv

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img_control)
axes[0].set_title(f"Control Group (Active)\n[{CH_BASE_NAME}]", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_ablated)
axes[1].set_title(f"Ablated Group (Inactive)\n[{CH_BASE_NAME}]", fontsize=12)
axes[1].axis("off")

plt.tight_layout()

comparison_plot_filename = f"ablation_comparison_{CH_BASE_NAME}.png"
plt.savefig(comparison_plot_filename, dpi=150, bbox_inches='tight')
print(f"Saved side-by-side comparison plot as: {comparison_plot_filename}")

plt.show()

assert not isinstance(model.blocks[8].ffn.dwconv, nn.Identity), "Restore failed!"
assert not isinstance(model.noise_refiner[0].ffn.dwconv, nn.Identity), "Restore failed!"

In [ ]:
weight_tensor = model.blocks[8].ffn.dwconv.weight.detach().cpu().to(torch.float32)
weight_tensor = weight_tensor.squeeze(1)

dirac_ref = torch.zeros_like(weight_tensor)
center = weight_tensor.shape[-1] // 2
dirac_ref[:, center, center] = 1.0

drift = (weight_tensor - dirac_ref).abs().mean(dim=(1, 2))

top_16_indices = torch.argsort(drift, descending=True)[:16]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle(f"Top 16 Learned Conv2D Filter Kernels (Block 8)\n[Sorted by Drift-From-Init | {CH_BASE_NAME}]", fontsize=14, y=0.98)

for i, idx in enumerate(top_16_indices):
    ax = axes[i // 4, i % 4]
    kernel_weights = weight_tensor[idx].numpy()
    
    vmax = max(abs(kernel_weights.min()), abs(kernel_weights.max()))
    im = ax.imshow(kernel_weights, cmap="coolwarm", interpolation="nearest", vmin=-vmax, vmax=vmax)
    
    ax.set_title(f"Channel {idx.item()}\nDrift: {drift[idx].item():.5f}\nVar: {torch.var(weight_tensor[idx]).item():.5f}", fontsize=8)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()

weight_plot_filename = f"weights_drift_{CH_BASE_NAME}.png"
plt.savefig(weight_plot_filename, dpi=150, bbox_inches='tight')
print(f"Saved active weight drift visualization as: {weight_plot_filename}")

plt.show()

In [ ]:
total_drift = 0.0
layer_count = 0

for name, block_list in [("blocks", model.blocks), ("noise_refiner", model.noise_refiner)]:
    for i, block in enumerate(block_list):
        if hasattr(block, 'ffn') and hasattr(block.ffn, 'dwconv'):
            w = block.ffn.dwconv.weight.detach().cpu().to(torch.float32).squeeze(1)
            
            ref = torch.zeros_like(w)
            c = w.shape[-1] // 2
            ref[:, c, c] = 1.0
            
            layer_drift = (w - ref).abs().mean().item()
            total_drift += layer_drift
            layer_count += 1

global_average_drift = total_drift / layer_count

print("="*50)
print(f"     GLOBAL CONVERGENCE DIAGNOSTIC: {CH_BASE_NAME}     ")
print("="*50)
print(f"Analyzed Convolutional Layers : {layer_count}")
print(f"Global Average Weight Drift   : {global_average_drift:.8f}")
print("="*50)
print("\n[DIAGNOSTIC CRITERIA]")
print("Compare this value between your checkpoints:")
print(" - If Drift(1000) > Drift(500) : The network is still learning. Extend training.")
print(" - If Drift(1000) == Drift(500): The network has fully converged on your subset.")

In [ ]:
intercepted_activations = {}

def hook_fn(module, input_args, output_tensor):
    intercepted_activations['before'] = input_args[0].detach().cpu().to(torch.float32)
    intercepted_activations['after'] = output_tensor.detach().cpu().to(torch.float32)

hook_target = model.blocks[8].ffn.dwconv
handle = hook_target.register_forward_hook(hook_fn)

print("Executing a single forward step to extract activations...")
dummy_t = torch.tensor([0.5], device=DEVICE).to(Config.dtype)
with torch.no_grad():
    _ = model(x=initial_noise.clone(), t=dummy_t, text_embeds=text_embeds, text_mask=text_mask)

handle.remove()

def project_channels_to_rgb(tensor_bchw):
    tensor_chw = tensor_bchw[0]
    C, H, W = tensor_chw.shape
    flat_features = tensor_chw.view(C, -1).t()
    centered_features = flat_features - flat_features.mean(dim=0, keepdim=True)
    U, S, V = torch.linalg.svd(centered_features, full_matrices=False)
    projected = torch.matmul(centered_features, V[:3, :].t())
    min_vals = projected.min(dim=0, keepdim=True).values
    max_vals = projected.max(dim=0, keepdim=True).values
    normalized_proj = (projected - min_vals) / (max_vals - min_vals + 1e-6)
    return normalized_proj.view(H, W, 3).numpy()

if 'before' in intercepted_activations and 'after' in intercepted_activations:
    print("Projecting latent feature spaces using SVD...")
    rgb_before = project_channels_to_rgb(intercepted_activations['before'])
    rgb_after = project_channels_to_rgb(intercepted_activations['after'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(rgb_before)
    axes[0].set_title(f"Latent Features BEFORE Conv2D (Pure MLP/SwiGLU)\n[{CH_BASE_NAME}]", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(rgb_after)
    axes[1].set_title(f"Latent Features AFTER Conv2D (Spatial Filtered)\n[{CH_BASE_NAME}]", fontsize=12)
    axes[1].axis("off")
    
    plt.tight_layout()
    
    svd_plot_filename = f"svd_activations_{CH_BASE_NAME}.png"
    plt.savefig(svd_plot_filename, dpi=150, bbox_inches='tight')
    print(f"Saved activation map visualization as: {svd_plot_filename}")
    
    plt.show()
else:
    print("Error: Hook failed to capture activations.")

In [ ]:
from samplers import get_schedule, euler_step, get_1d_shifted_time
import math

def capture_trajectory_steps(model, initial_noise, steps, combined_text_embeds, cfg=2.5, text_mask=None, shift_val=2.5):
    device = initial_noise.device
    raw_timesteps = get_schedule(Config.validate_scheduler, steps, device)
    timesteps = get_1d_shifted_time(raw_timesteps, shift_val)
    
    x = initial_noise.clone().to(device=DEVICE, dtype=Config.dtype)
    captured_frames = {}
    
    target_steps = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 99]
    
    for i in range(steps):
        t = timesteps[i].view(-1)
        t_next = timesteps[i+1].view(-1)
        dt = t_next - t
        
        if i in target_steps:
            pct = int((i / steps) * 100)
            captured_frames[pct] = x.clone()
            
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                x = euler_step(model, x, t, dt, combined_text_embeds, cfg, text_mask)
                x = x.to(dtype=Config.dtype)
                
    captured_frames[100] = x.clone()
    return captured_frames

print("Tracing the transport flow trajectory (100 steps)...")
frames = capture_trajectory_steps(
    model, 
    initial_noise, 
    steps=100, 
    combined_text_embeds=combined_text_embeds, 
    cfg=2.5,
    text_mask=combined_mask, 
    shift_val=2.5
)

decoded_frames = []
percentages = sorted(list(frames.keys()))

for pct in percentages:
    print(f"Decoding path at {pct}%...")
    img = decode_latents_to_image(vae, frames[pct], DEVICE)
    decoded_frames.append((pct, img))

num_plots = len(decoded_frames)
cols = 5
rows = math.ceil(num_plots / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows), squeeze=False)
axes_flat = axes.flatten()

for i, (pct, img) in enumerate(decoded_frames):
    ax = axes_flat[i]
    ax.imshow(img)
    ax.set_title(f"Trajectory: {pct}% of Flow", fontsize=10)
    ax.axis("off")

for j in range(num_plots, len(axes_flat)):
    axes_flat[j].axis("off")

plt.tight_layout()
trajectory_plot_filename = f"flow_trajectory_{CH_BASE_NAME}.png"
plt.savefig(trajectory_plot_filename, dpi=150, bbox_inches='tight')
print(f"\nSaved trajectory static grid as: {trajectory_plot_filename}")
plt.show()

In [ ]:
print("Compiling frames into an animated GIF...")

gif_frames = [img for _, img in decoded_frames]

gif_filename = f"flow_trajectory_animation_{CH_BASE_NAME}.gif"

gif_frames[0].save(
    gif_filename,
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
    lossless=True
)

print(f"\nSuccess! Smooth animated trajectory saved to: {gif_filename}")

In [ ]:
file_A = "10003724.pt"
target_file_A = os.path.join(Config.cache_dir, file_A)  

if not os.path.exists(target_file_A):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_A}")

file_B = "10005361.pt"
target_file_B = os.path.join(Config.cache_dir, file_B)  

if not os.path.exists(target_file_B):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_B}")

print(f"Blending Concept A ({os.path.basename(file_A)}) with Concept B ({os.path.basename(file_B)})...")

data_A = torch.load(target_file_A, map_location=DEVICE)
data_B = torch.load(target_file_B, map_location=DEVICE)

h_A, w_A = int(data_A["height"]), int(data_A["width"])
h_B, w_B = int(data_B["height"]), int(data_B["width"])

h, w = h_A, w_A

def get_cond(data):
    if "text_embeds_list" in data:
        return data["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
    return data["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask"].unsqueeze(0).to(DEVICE)

embed_A, mask_A = get_cond(data_A)
embed_B, mask_B = get_cond(data_B)

max_len = max(embed_A.shape[1], embed_B.shape[1])
def pad_embed(embed, mask, target_len):
    if embed.shape[1] < target_len:
        pad_size = target_len - embed.shape[1]
        embed = torch.cat([embed, torch.zeros(1, pad_size, embed.shape[2], device=DEVICE, dtype=Config.dtype)], dim=1)
        mask = torch.cat([mask, torch.zeros(1, pad_size, device=DEVICE, dtype=torch.bool)], dim=1)
    return embed, mask

embed_A, mask_A = pad_embed(embed_A, mask_A, max_len)
embed_B, mask_B = pad_embed(embed_B, mask_B, max_len)

gen_A = torch.Generator(device=DEVICE).manual_seed(100)
gen_B = torch.Generator(device=DEVICE).manual_seed(200)
noise_A = torch.randn(1, Config.in_channels, h // Config.vae_downsample_factor, w // Config.vae_downsample_factor, generator=gen_A, device=DEVICE, dtype=Config.dtype)
noise_B = torch.randn(1, Config.in_channels, h // Config.vae_downsample_factor, w // Config.vae_downsample_factor, generator=gen_B, device=DEVICE, dtype=Config.dtype)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
morph_images = []

for alpha in alphas:
    print(f"Rendering frame for Alpha = {alpha:.2f}...")
    
    blended_noise = (1.0 - alpha) * noise_A + alpha * noise_B
    blended_embed = (1.0 - alpha) * embed_A + alpha * embed_B
    blended_mask = mask_A | mask_B 
    
    uncond_embed = torch.zeros_like(blended_embed)
    uncond_mask = torch.ones_like(blended_mask)
    comb_embed = torch.cat([uncond_embed, blended_embed], dim=0)
    comb_mask = torch.cat([uncond_mask, blended_mask], dim=0)
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            latents = run_sampling_pipeline(
                model=model, 
                initial_noise=blended_noise, 
                steps=30, 
                combined_text_embeds=comb_embed, 
                cfg=2.5,
                text_mask=comb_mask,
                sampler_type="euler",
                scheduler_type="uniform",
                shift_val=2.5
            )
            
    img = decode_latents_to_image(vae, latents, DEVICE)
    morph_images.append((alpha, img))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f"Conceptual Manifold Morphing (Concept A -> Concept B)\n[CFG = 2.5 | {CH_BASE_NAME}]", fontsize=14, y=0.98)

for i, (alpha, img) in enumerate(morph_images):
    axes[i].imshow(img)
    axes[i].set_title(f"Alpha: {alpha:.2f}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
morph_plot_filename = f"conceptual_morph_{CH_BASE_NAME}.png"
plt.savefig(morph_plot_filename, dpi=150, bbox_inches='tight')
print(f"Saved conceptual morphing strip as: {morph_plot_filename}")
plt.show()

In [ ]:
cfg_values = [0.0, 1.0, 2.5, 5.0, 10.0]
sweep_images = []

print("Running Classifier-Free Guidance (CFG) sweep...")
for cfg in cfg_values:
    print(f"Generating image with CFG = {cfg:.1f}...")
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            latents_cfg = run_sampling_pipeline(
                model=model, 
                initial_noise=initial_noise.clone(), 
                steps=30, 
                combined_text_embeds=combined_text_embeds, 
                cfg=cfg, 
                text_mask=combined_mask,
                sampler_type="euler",
                scheduler_type="uniform",
                shift_val=2.5
            )
            
    img = decode_latents_to_image(vae, latents_cfg, DEVICE)
    sweep_images.append((cfg, img))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f"Classifier-Free Guidance (CFG) Saturation Sweep\n[{CH_BASE_NAME}]", fontsize=14, y=0.98)

for i, (cfg, img) in enumerate(sweep_images):
    axes[i].imshow(img)
    axes[i].set_title(f"CFG: {cfg:.1f}", fontsize=12)
    axes[i].axis("off")

plt.tight_layout()
cfg_sweep_filename = f"cfg_sweep_{CH_BASE_NAME}.png"
plt.savefig(cfg_sweep_filename, dpi=150, bbox_inches='tight')
print(f"Saved CFG sweep visualization as: {cfg_sweep_filename}")
plt.show()

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from latents import normalize_latents, decode_latents_to_image
from samplers import euler_step, get_1d_shifted_time

target_file_A = "10003724.pt"
target_file_A = os.path.join(Config.cache_dir, target_file_A)

if not os.path.exists(target_file_A):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_A}")

target_file_B = "5567876.pt"
target_file_B = os.path.join(Config.cache_dir, target_file_B)

if not os.path.exists(target_file_B):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_B}")

data_A = torch.load(target_file_A, map_location=DEVICE)
data_B = torch.load(target_file_B, map_location=DEVICE)

h_A, w_A = int(data_A["height"]), int(data_A["width"])

print(f"Extracting and normalizing structure latent from Concept A ({target_file_A})...")
clean_latent = normalize_latents(data_A["latents"].unsqueeze(0).to(DEVICE, Config.dtype))

noise_strength = 0.95
start_t = 1.0 - noise_strength

print(f"Injecting {noise_strength*100:.0f}% Gaussian noise...")
torch_generator = torch.Generator(device=DEVICE).manual_seed(42)
pure_noise = torch.randn_like(clean_latent, generator=torch_generator)

partially_noisy_latent = (1.0 - start_t) * pure_noise + start_t * clean_latent

def get_cond(data):
    if "text_embeds_list" in data:
        return data["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
    return data["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask"].unsqueeze(0).to(DEVICE)

embed_B, mask_B = get_cond(data_B)

uncond_B = torch.zeros_like(embed_B)
uncond_mask_B = torch.ones_like(mask_B)
comb_embed_B = torch.cat([uncond_B, embed_B], dim=0)
comb_mask_B = torch.cat([uncond_mask_B, mask_B], dim=0)

SHIFT_VAL = 2.5
steps = 20

def invert_shifted_time(t, shift_val):
    s = 1.0 / shift_val
    return t / (s * (1.0 - t) + t)

u_start = invert_shifted_time(torch.tensor(start_t, device=DEVICE, dtype=torch.float32), SHIFT_VAL)
u_schedule = torch.linspace(u_start.item(), 1.0, steps + 1, device=DEVICE)
timesteps = get_1d_shifted_time(u_schedule, shift_val=SHIFT_VAL)

x = partially_noisy_latent.clone()

print(f"Denoising toward Concept B ({target_file_B}) using CFG = 2.5...")
for i in range(steps):
    t = timesteps[i].view(-1)
    t_next = timesteps[i + 1].view(-1)
    dt = t_next - t

    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            x = euler_step(model, x, t, dt, comb_embed_B, cfg=2.5, text_mask=comb_mask_B)
            x = x.to(dtype=Config.dtype)

print("Decoding latents to image space...")
img_control_A = decode_latents_to_image(vae, clean_latent, DEVICE)
img_noisy_start = decode_latents_to_image(vae, partially_noisy_latent, DEVICE)
img_translated = decode_latents_to_image(vae, x, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_control_A)
axes[0].set_title(f"Original Concept A Structure\n[{os.path.basename(target_file_A)}]", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_noisy_start)
axes[1].set_title(f"Noised Input ({noise_strength*100:.0f}% Noise)\n[BEFORE any denoising — check this panel]", fontsize=12)
axes[1].axis("off")

axes[2].imshow(img_translated)
axes[2].set_title(f"SDEdit Style-Translated Output\n(Prompted to {os.path.basename(target_file_B)} | CFG = 2.5)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
CH_BASE_NAME = f"{os.path.splitext(os.path.basename(target_file_A))[0]}_to_{os.path.splitext(os.path.basename(target_file_B))[0]}"
sdedit_filename = f"sdedit_translation_{CH_BASE_NAME}.png"
plt.savefig(sdedit_filename, dpi=150, bbox_inches='tight')
print(f"Saved SDEdit style-translation as: {sdedit_filename}")
plt.show()

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms.functional as TF
from text_encoder import TextEncoderWrapper
from latents import normalize_latents, decode_latents_to_image
from samplers import euler_step, get_1d_shifted_time

UNSEEN_IMAGE_PATH = r"C:\Users\Leonardo\Desktop\image.webp"
CUSTOM_PROMPT = "no humans, star (sky), full moon, night sky, scenery, outdoors, purple flower, cloud, landscape, starry sky, grass, mountain, mountainous horizon, nature, plant"
NOISE_STRENGTH = 0.75

if 'text_encoder' not in globals():
    print("Loading Qwen3 Text Encoder into VRAM...")
    text_encoder = TextEncoderWrapper(dtype=Config.dtype, device=DEVICE)

if not os.path.exists(UNSEEN_IMAGE_PATH):
    raise FileNotFoundError(f"Please place an image at '{UNSEEN_IMAGE_PATH}' or change the path in the cell.")

print(f"Loading and cropping: {UNSEEN_IMAGE_PATH}")
raw_img = Image.open(UNSEEN_IMAGE_PATH).convert("RGB")

bw, bh = Config.target_resolution, Config.target_resolution
img_w, img_h = raw_img.size
img_aspect = img_w / img_h
target_aspect = bw / bh
resize_w, resize_h = (int(bh * img_aspect), bh) if img_aspect > target_aspect else (bw, int(bw / img_aspect))
resized_img = raw_img.resize((resize_w, resize_h), resample=Image.LANCZOS)
left, top = (resize_w - bw) // 2, (resize_h - bh) // 2
cropped_img = resized_img.crop((left, top, left + bw, top + bh))

img_tensor = TF.to_tensor(cropped_img).unsqueeze(0).to(DEVICE, Config.dtype)
batch_tensor_vae = TF.normalize(img_tensor, [0.5], [0.5])

print("Compressing image into VAE Latent Space...")
with torch.no_grad():
    with torch.amp.autocast("cuda", enabled=False):
        encoded = vae.encode(batch_tensor_vae.float())
        latents_raw = encoded.latent_dist.mode() if hasattr(encoded, "latent_dist") else encoded[0]

clean_latent = normalize_latents(latents_raw.to(Config.dtype))

start_t = 1.0 - NOISE_STRENGTH
print(f"Injecting {NOISE_STRENGTH*100:.0f}% noise...")
torch_generator = torch.Generator(device=DEVICE).manual_seed(42)
pure_noise = torch.randn_like(clean_latent, generator=torch_generator)

partially_noisy_latent = (1.0 - start_t) * pure_noise + start_t * clean_latent

print(f"Encoding custom prompt: '{CUSTOM_PROMPT}'")
with torch.no_grad():
    embed, mask = text_encoder.encode([CUSTOM_PROMPT])

mask_bool = mask.bool().to(DEVICE)
uncond = torch.zeros_like(embed)
uncond_mask = torch.ones_like(mask_bool, dtype=torch.bool)
comb_embed = torch.cat([uncond, embed], dim=0)
comb_mask = torch.cat([uncond_mask, mask_bool], dim=0)

SHIFT_VAL = 2.5
steps = 100

def invert_shifted_time(t, shift_val):
    s = 1.0 / shift_val
    return t / (s * (1.0 - t) + t)

u_start = invert_shifted_time(torch.tensor(start_t, device=DEVICE, dtype=torch.float32), SHIFT_VAL)
u_schedule = torch.linspace(u_start.item(), 1.0, steps + 1, device=DEVICE)
timesteps = get_1d_shifted_time(u_schedule, shift_val=SHIFT_VAL)

x = partially_noisy_latent.clone()

print("Denoising toward custom prompt using CFG = 1.0...")
for i in range(steps):
    t = timesteps[i].view(-1)
    t_next = timesteps[i + 1].view(-1)
    dt = t_next - t

    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            x = euler_step(model, x, t, dt, comb_embed, cfg=1.0, text_mask=comb_mask)
            x = x.to(dtype=Config.dtype)

print("Decoding results...")
img_noisy_start = decode_latents_to_image(vae, partially_noisy_latent, DEVICE)
img_translated = decode_latents_to_image(vae, x, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cropped_img)
axes[0].set_title("Your Raw Input Image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_noisy_start)
axes[1].set_title(f"Noised Latent Input ({NOISE_STRENGTH*100:.0f}% Noise)\n[BEFORE any denoising — check this panel]", fontsize=12)
axes[1].axis("off")

axes[2].imshow(img_translated)
axes[2].set_title(f"Anime-ified Output\n(Prompted: '{CUSTOM_PROMPT}' | CFG = 1.0)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
img_base = os.path.splitext(os.path.basename(UNSEEN_IMAGE_PATH))[0]
universal_sdedit_filename = f"universal_translation_{img_base}.png"
plt.savefig(universal_sdedit_filename, dpi=150, bbox_inches='tight')
print(f"Saved style translation as: {universal_sdedit_filename}")
plt.show()

In [ ]:
print(vae.config)
print(getattr(vae.config, "in_channels", None), getattr(vae.config, "latent_channels", None))